# Testing Jev (TypeSafe)

Jev is TypeSafe's "System One" model. It doesn't generate text. You send it a **state** (your content) plus typed **questions**, and it returns structured answers:

- **noul**: yes/no, returns a probability from 0 to 1
- **choice**: picks one option from a list, with probabilities
- **score**: rates the content on a rubric, with probabilities

Docs: https://docs.typesafe.ai/introduction

This is **Part 1 of 3**: calling Jev with raw HTTP. Next: `part2_sdk.ipynb`.

Install the libraries once (from the repo folder): `uv pip install -r requirements.txt`, then pick the `.venv` kernel.

# Part 1: Raw HTTP with `requests`

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv, find_dotenv

In [ ]:
load_dotenv(find_dotenv(usecwd=True), override=True)
api_key = os.getenv("TYPESAFE_API_KEY") or os.getenv("JEV_API_KEY")

if api_key:
    print(f"API key found, starts with {api_key[:6]}")
else:
    print("TYPESAFE_API_KEY not set - add it to your .env file")

BASE_URL = "https://api.typesafe.ai/v1"
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

## 1. Check the key works and list available models

In [ ]:
response = requests.get(f"{BASE_URL}/models", headers=headers, timeout=30)
print(response.status_code)
print(json.dumps(response.json(), indent=2))

## 2. A helper to ask Jev questions about some state

In [ ]:
def ask_jev(state, questions, model="jev-latest"):
    body = {"model": model, "state": state, "questions": questions}
    response = requests.post(f"{BASE_URL}/systemone", headers=headers, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## 3. Ask one question of each type

In [ ]:
state = "I was charged twice for my subscription this month. Please refund the extra charge."

questions = {
    "billing": {
        "type": "noul",
        "instructions": "Is this message about billing?",
    },
    "tone": {
        "type": "choice",
        "instructions": "What is the tone of this message?",
        "criteria": {
            "angry": "Hostile or upset",
            "neutral": "Calm and matter-of-fact",
            "happy": "Positive",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this message?",
        "criteria": ["Not urgent", "Somewhat urgent", "Very urgent"],
    },
}

result = ask_jev(state, questions)
print(json.dumps(result, indent=2))

## 4. Read the answers

In [ ]:
answers = result["answers"]

print(f"Model used: {result['model']}")
print(f"Billing (yes probability): {answers['billing']['noul']}")
print(f"Tone: {answers['tone']['choice']} (confidence {answers['tone']['confidence']})")
print(f"Urgency score: {answers['urgency']['score']} (confidence {answers['urgency']['confidence']})")
print(f"Tokens: {result['usage']}")

## 5. Try your own

Change `my_state` and `my_questions`, then run the cell. The state can also be a dict (JSON).

In [ ]:
my_state = {"subject": "Server down", "message": "Production has been offline for 20 minutes, customers can't log in!"}

my_questions = {
    "is_outage": {"type": "noul", "instructions": "Does this describe a service outage?"},
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {"engineering": "Technical problems", "billing": "Payments and invoices", "sales": "New customers"},
    },
}

print(json.dumps(ask_jev(my_state, my_questions)["answers"], indent=2))